# BIC coursework: ANN + PSO
**How to use:** run the cells from top to bottom (click a cell, press Shift+Enter). Only Cell 1 needs you to do something: choose `data_banknote_authentication.txt` from your computer.

Colab deletes everything when the session ends, so run Cell 12 to download your results before you close it.

## Cell 1: upload the dataset (the ONLY upload)
Click **Choose files** and pick `data_banknote_authentication.txt` from your BIC---course-work folder.

In [ ]:
from google.colab import files
files.upload()


## Cells 2-8: create the code files (just run each cell)

In [ ]:
%%writefile pso.py
"""Task 2: Particle Swarm Optimisation with informants, from scratch.

Follows Algorithm 39 (Section 3.5, "Essentials of Metaheuristics", Luke).
Comments tag the corresponding algorithm line numbers as "Alg39 L<n>".

!! Before submitting, check the line numbers below against the lecture slides / book
!! copy you have; I wrote them from memory of the algorithm's structure.

Algorithm 39 (as used here):
 1 swarmsize   2 alpha (velocity retained)   3 beta (personal best)
 4 gamma (informants' best)   5 delta (global best)   6 epsilon (jump size)
 7 P <- {}   8-9 create random particles with random velocity   10 Best <- none
 11 repeat
 12-15   assess each particle; update Best
 16-24   for each particle: find x* (own best), x+ (informants' best, incl. self),
         x! (global best); for each dimension: draw b,c,d and update velocity
 25-26   move each particle: x <- x + epsilon * v
 27 until Best ideal or out of time        28 return Best

Fitness is MAXIMISED (higher = better), matching Algorithm 39.
"""
import numpy as np


class PSO:
    def __init__(self, dim, fitness_fn, swarm_size=30, iterations=100,
                 alpha=0.7, beta=1.5, gamma=1.5, delta=0.5, epsilon=1.0,
                 n_informants=3, pos_range=(-1.0, 1.0), boundary="clip",
                 resample_informants=False, seed=None):
        self.dim = dim
        self.fitness_fn = fitness_fn
        self.swarm_size = swarm_size            # Alg39 L1
        self.iterations = iterations
        self.alpha, self.beta = alpha, beta     # Alg39 L2, L3
        self.gamma, self.delta = gamma, delta   # Alg39 L4, L5
        self.epsilon = epsilon                  # Alg39 L6
        self.n_informants = n_informants        # informants per particle (excl. self)
        self.lo, self.hi = pos_range
        self.boundary = boundary                # "none" | "clip" | "reflect"
        self.resample_informants = resample_informants
        self.rng = np.random.default_rng(seed)

    # ---- informant topology -------------------------------------------------
    def _make_informants(self):
        """Each particle is informed by itself + n_informants random others."""
        S = self.swarm_size
        k = min(self.n_informants, S - 1)
        inf = []
        for i in range(S):
            others = [j for j in range(S) if j != i]
            chosen = self.rng.choice(others, size=k, replace=False)
            inf.append(np.append(chosen, i))
        return inf

    def _handle_boundary(self, X, V):
        if self.boundary == "none":
            return X, V
        if self.boundary == "clip":             # stick to wall, kill velocity there
            out = (X < self.lo) | (X > self.hi)
            X = np.clip(X, self.lo, self.hi)
            V = np.where(out, 0.0, V)
        elif self.boundary == "reflect":        # bounce back, reverse velocity
            span = self.hi - self.lo
            for _ in range(2):  # handles overshoot up to ~1 span
                over, under = X > self.hi, X < self.lo
                X = np.where(over, 2 * self.hi - X, X)
                X = np.where(under, 2 * self.lo - X, X)
                V = np.where(over | under, -V, V)
            X = np.clip(X, self.lo, self.hi)
        else:
            raise ValueError(f"Unknown boundary handling: {self.boundary}")
        return X, V

    def run(self):
        S, D = self.swarm_size, self.dim
        span = self.hi - self.lo

        # Alg39 L7-9: random particles with random initial velocities
        X = self.rng.uniform(self.lo, self.hi, (S, D))
        V = self.rng.uniform(-0.1 * span, 0.1 * span, (S, D))

        pbest_X = X.copy()                      # "previous fittest location" of each particle
        pbest_f = np.full(S, -np.inf)
        best_x, best_f = None, -np.inf          # Alg39 L10: Best <- none
        informants = self._make_informants()
        history = []                            # best fitness after each iteration
        evals = 0

        for _ in range(self.iterations):        # Alg39 L11 (repeat ... until time out L27)
            # Alg39 L12-15: assess fitness, track Best (and each particle's own best)
            for i in range(S):
                f = self.fitness_fn(X[i])
                evals += 1
                if f > pbest_f[i]:
                    pbest_f[i], pbest_X[i] = f, X[i].copy()
                if f > best_f:
                    best_f, best_x = f, X[i].copy()
            history.append(best_f)

            if self.resample_informants:
                informants = self._make_informants()

            # Alg39 L16-24: velocity update
            g_idx = int(np.argmax(pbest_f))
            x_global = pbest_X[g_idx]           # Alg39 L19: x! (best of any particle)
            newV = np.empty_like(V)
            for i in range(S):
                x_star = pbest_X[i]             # Alg39 L17: x* own best
                inf = informants[i]
                x_plus = pbest_X[inf[np.argmax(pbest_f[inf])]]  # Alg39 L18: x+
                b = self.rng.uniform(0, self.beta, D)    # Alg39 L21
                c = self.rng.uniform(0, self.gamma, D)   # Alg39 L22
                d = self.rng.uniform(0, self.delta, D)   # Alg39 L23
                newV[i] = (self.alpha * V[i]             # Alg39 L24
                           + b * (x_star - X[i])
                           + c * (x_plus - X[i])
                           + d * (x_global - X[i]))
            V = newV

            # Alg39 L25-26: move particles
            X = X + self.epsilon * V
            X, V = self._handle_boundary(X, V)  # extension (not in Alg 39)

        return best_x, best_f, history, evals   # Alg39 L28


In [ ]:
%%writefile data.py
"""Task 4 data handling: load banknote CSV, stratified 70/30 split, standardise."""
import numpy as np


def load_banknote(path="data_banknote_authentication.txt"):
    """Rows: 4 features, then label (0/1). Comma separated, no header."""
    data = np.loadtxt(path, delimiter=",")
    return data[:, :4], data[:, 4].astype(int)


def stratified_split(X, y, test_frac=0.3, seed=0):
    """Keeps the class ratio the same in train and test."""
    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for cls in np.unique(y):
        idx = rng.permutation(np.where(y == cls)[0])
        n_test = int(round(len(idx) * test_frac))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])
    train_idx, test_idx = rng.permutation(train_idx), rng.permutation(test_idx)
    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]


def standardise(X_train, X_test):
    """z-score using TRAIN statistics only (avoids test-set leakage)."""
    mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
    sd[sd == 0] = 1.0
    return (X_train - mu) / sd, (X_test - mu) / sd


In [ ]:
%%writefile ann.py
"""Task 1: Feedforward multi-layer ANN, implemented from scratch (NumPy only for maths).

No training algorithm lives here. The ANN exposes its weights and biases as ONE flat
vector (get/set_params) so that PSO can optimise them (see run.py for the coupling).
"""
import numpy as np


# ---- Activation functions (required: logistic, ReLU, tanh) -------------------
def logistic(x):
    # clip to avoid overflow in exp for large |x|
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def relu(x):
    return np.maximum(0.0, x)


def tanh(x):
    return np.tanh(x)


ACTIVATIONS = {"logistic": logistic, "relu": relu, "tanh": tanh}


class ANN:
    """Fully connected feedforward network.

    layer_sizes: e.g. [4, 8, 1]  -> 4 inputs, one hidden layer of 8, 1 output
    activations: one name per NON-input layer, e.g. ["tanh", "logistic"]
    """

    def __init__(self, layer_sizes, activations):
        if len(activations) != len(layer_sizes) - 1:
            raise ValueError("Need one activation per non-input layer")
        self.layer_sizes = list(layer_sizes)
        self.activations = list(activations)
        # Parameter count: for each layer, (n_in * n_out) weights + n_out biases
        self.n_params = sum(
            n_in * n_out + n_out
            for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])
        )
        self.weights = []
        self.biases = []
        self.set_params(np.zeros(self.n_params))

    def set_params(self, vec):
        """Unpack a flat vector into per-layer weight matrices and bias vectors."""
        vec = np.asarray(vec, dtype=float)
        expected = getattr(self, "n_weight_params", self.n_params)  # subclass adds extra genes
        if vec.size != expected:
            raise ValueError(f"Expected {expected} params, got {vec.size}")
        self.weights, self.biases = [], []
        pos = 0
        for n_in, n_out in zip(self.layer_sizes[:-1], self.layer_sizes[1:]):
            w = vec[pos:pos + n_in * n_out].reshape(n_in, n_out)
            pos += n_in * n_out
            b = vec[pos:pos + n_out]
            pos += n_out
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, X):
        """X: (n_samples, n_features) -> (n_samples, n_outputs)."""
        a = np.asarray(X, dtype=float)
        for w, b, act in zip(self.weights, self.biases, self.activations):
            a = ACTIVATIONS[act](a @ w + b)
        return a

    def predict(self, X, threshold=None):
        """Binary class labels from a single output node.
        threshold=None picks 0.0 for a tanh output and 0.5 otherwise."""
        if threshold is None:
            threshold = 0.0 if self.activations[-1] == "tanh" else 0.5
        return (self.forward(X)[:, 0] >= threshold).astype(int)


class ANNLearnedActs(ANN):
    """EXTENSION: the hidden-layer activation functions are part of the PSO vector.

    Particle layout: [all weights and biases ... | one gene per hidden layer].
    Each extra gene g in [-r, r] is decoded to logistic / relu / tanh by splitting
    [-r, r] into three equal bins. The output layer's activation is fixed.
    """
    CHOICES = ["logistic", "relu", "tanh"]

    def __init__(self, layer_sizes, output_activation="logistic", act_range=2.0):
        self._ready = False                     # lets the base __init__ call set_params
        self.act_range = act_range
        self.n_hidden = len(layer_sizes) - 2
        super().__init__(layer_sizes, ["tanh"] * self.n_hidden + [output_activation])
        self._ready = True
        self.n_weight_params = self.n_params
        self.n_params += self.n_hidden
        self.set_params(np.zeros(self.n_params))

    def decode_act(self, gene):
        r = self.act_range
        frac = (np.clip(gene, -r, r) + r) / (2 * r)
        return self.CHOICES[min(int(frac * len(self.CHOICES)), len(self.CHOICES) - 1)]

    def set_params(self, vec):
        vec = np.asarray(vec, dtype=float)
        if not self._ready:
            return super().set_params(vec)
        if vec.size != self.n_params:
            raise ValueError(f"Expected {self.n_params} params, got {vec.size}")
        super().set_params(vec[:self.n_weight_params])
        for layer, gene in enumerate(vec[self.n_weight_params:]):
            self.activations[layer] = self.decode_act(gene)


In [ ]:
%%writefile run.py
"""Task 3 + 4: couple PSO to the ANN and train/evaluate on the banknote data.

Coupling: each PSO particle is a flat vector of length ann.n_params. To evaluate a
particle we ann.set_params(particle) and measure the ANN's accuracy on the TRAINING set;
that accuracy is the particle's fitness. The test set is only used once at the end.

Single configuration from the command line:
    python run.py --layers 4 4 1 --acts tanh logistic --swarm 20 --iters 25 --runs 10
Learned hidden activations (extension):
    python run.py --layers 4 4 1 --learn_acts
For many configurations use sweep.py.
"""
import argparse
import csv
import numpy as np

from ann import ANN, ANNLearnedActs
from pso import PSO
from data import load_banknote, stratified_split, standardise

DEFAULTS = dict(layers=[4, 4, 1], acts=["tanh", "logistic"], learn_acts=False,
                swarm=20, iters=25, alpha=0.7, beta=1.5, gamma=1.5, delta=0.5,
                epsilon=1.0, informants=3, resample=False, range=2.0,
                boundary="clip", metric="accuracy")


def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))


def make_fitness(ann, X, y, metric="accuracy"):
    """Build the fitness function PSO maximises (the coupling point)."""
    def fitness(vec):
        ann.set_params(vec)
        if metric == "accuracy":
            return accuracy(y, ann.predict(X))
        if metric == "neg_mse":                 # smoother landscape than accuracy
            return -float(np.mean((ann.forward(X)[:, 0] - y) ** 2))
        raise ValueError(metric)
    return fitness


def prepare_split(path, split_seed=0):
    X, y = load_banknote(path)
    Xtr, ytr, Xte, yte = stratified_split(X, y, 0.3, split_seed)
    Xtr, Xte = standardise(Xtr, Xte)
    return Xtr, ytr, Xte, yte


def one_run(cfg, seed, split):
    """One PSO training run. cfg: dict like DEFAULTS. Returns a result dict."""
    cfg = {**DEFAULTS, **cfg}
    Xtr, ytr, Xte, yte = split
    if cfg["learn_acts"]:
        ann = ANNLearnedActs(cfg["layers"], cfg["acts"][-1], act_range=cfg["range"])
    else:
        ann = ANN(cfg["layers"], cfg["acts"])
    pso = PSO(dim=ann.n_params, fitness_fn=make_fitness(ann, Xtr, ytr, cfg["metric"]),
              swarm_size=cfg["swarm"], iterations=cfg["iters"],
              alpha=cfg["alpha"], beta=cfg["beta"], gamma=cfg["gamma"],
              delta=cfg["delta"], epsilon=cfg["epsilon"],
              n_informants=cfg["informants"], pos_range=(-cfg["range"], cfg["range"]),
              boundary=cfg["boundary"], resample_informants=cfg["resample"], seed=seed)
    best_x, best_f, history, evals = pso.run()
    ann.set_params(best_x)
    return dict(train=accuracy(ytr, ann.predict(Xtr)), test=accuracy(yte, ann.predict(Xte)),
                evals=evals, history=[float(h) for h in history],
                acts=list(ann.activations))


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="data_banknote_authentication.txt")
    p.add_argument("--layers", type=int, nargs="+", default=DEFAULTS["layers"])
    p.add_argument("--acts", nargs="+", default=DEFAULTS["acts"])
    p.add_argument("--learn_acts", action="store_true",
                   help="encode hidden activations in the particle (extension)")
    for k in ["alpha", "beta", "gamma", "delta", "epsilon", "range"]:
        p.add_argument(f"--{k}", type=float, default=DEFAULTS[k])
    p.add_argument("--swarm", type=int, default=DEFAULTS["swarm"])
    p.add_argument("--iters", type=int, default=DEFAULTS["iters"])
    p.add_argument("--informants", type=int, default=DEFAULTS["informants"])
    p.add_argument("--resample", action="store_true", help="re-draw informants each iteration")
    p.add_argument("--boundary", default="clip", choices=["none", "clip", "reflect"])
    p.add_argument("--metric", default="accuracy", choices=["accuracy", "neg_mse"])
    p.add_argument("--runs", type=int, default=10)
    p.add_argument("--split_seed", type=int, default=0)
    p.add_argument("--out", default=None, help="CSV to append per-run results to")
    a = p.parse_args()
    cfg = {k: getattr(a, k) for k in DEFAULTS}

    split = prepare_split(a.data, a.split_seed)
    rows = []
    for r in range(a.runs):
        res = one_run(cfg, r, split)
        rows.append((r, res))
        print(f"run {r}: train acc {res['train']:.4f}  test acc {res['test']:.4f}  "
              f"evals {res['evals']}" + (f"  acts {res['acts']}" if a.learn_acts else ""))
    tr = np.array([x[1]["train"] for x in rows]); te = np.array([x[1]["test"] for x in rows])
    sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
    print(f"TRAIN mean {tr.mean():.4f} sd {sd(tr):.4f} | TEST mean {te.mean():.4f} sd {sd(te):.4f}")
    if a.out:
        with open(a.out, "a", newline="") as f:
            w = csv.writer(f)
            for r, res in rows:
                w.writerow([cfg, r, res["train"], res["test"], res["evals"]])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sweep.py
"""Run a named experiment (a grid of configurations x N runs) and save the results.

    python sweep.py list
    python sweep.py budget            # one experiment
    python sweep.py all --runs 10     # everything
    python sweep.py arch --workers 4

Outputs in ./results/: <exp>_summary.csv (mean/sd per config) and <exp>_hist.json
(best fitness per iteration for every run, for convergence plots).

The grids below are STARTING POINTS. Choosing the ranges, justifying them from the
literature and interpreting the results is your job (and must be your own work).
"""
import argparse
import csv
import json
import os
from multiprocessing import Pool

import numpy as np

from run import DEFAULTS, one_run, prepare_split

DATA = "data_banknote_authentication.txt"
H = lambda hidden, act: dict(layers=[4] + hidden + [1], acts=[act] * len(hidden) + ["logistic"])

EXPERIMENTS = {
    # Q1: architecture (layers, neurons, activation)
    "arch": {f"{h}-{a}": H(h, a) for h in ([1], [2], [4], [8], [4, 4], [8, 8])
             for a in ("logistic", "relu", "tanh")},
    # Q2: fixed budget of 500 evaluations split different ways
    "budget": {f"swarm{s}x{500 // s}it": dict(swarm=s, iters=500 // s)
               for s in (5, 10, 20, 25, 50, 100)},
    # Q3: cognitive (beta) vs social (gamma) balance, and the global term (delta)
    "coef": {**{f"b{b}_g{g}": dict(beta=b, gamma=g)
                for b, g in ((0.5, 2.5), (1.0, 2.0), (1.5, 1.5), (2.0, 1.0), (2.5, 0.5))},
             **{f"delta{d}": dict(delta=d) for d in (0.0, 0.5, 1.0, 2.0)}},
    # extras
    "informants": {f"inf{k}": dict(informants=k) for k in (1, 2, 3, 5, 10)},
    "boundary": {b: dict(boundary=b) for b in ("none", "clip", "reflect")},
    "learned_acts": {"fixed-tanh": H([4], "tanh"),
                     "learned": dict(layers=[4, 4, 1], acts=["tanh", "logistic"], learn_acts=True),
                     "fixed-relu": H([4], "relu"), "fixed-logistic": H([4], "logistic")},
    "metric": {m: dict(metric=m) for m in ("accuracy", "neg_mse")},
}

_split = None


def _init(split):
    global _split
    _split = split


def _job(args):
    name, cfg, seed = args
    return name, seed, one_run(cfg, seed, _split)


def run_experiment(exp, runs, workers, data):
    grid = EXPERIMENTS[exp]
    jobs = [(n, c, s) for n, c in grid.items() for s in range(runs)]
    split = prepare_split(data)       # load in the main process so a bad path fails fast
    with Pool(workers, initializer=_init, initargs=(split,)) as pool:
        out = pool.map(_job, jobs)
    os.makedirs("results", exist_ok=True)
    hist, rows = {}, []
    for name in grid:
        rs = [r for n, s, r in sorted(out, key=lambda t: (t[0], t[1])) if n == name]
        tr, te = np.array([r["train"] for r in rs]), np.array([r["test"] for r in rs])
        hist[name] = [r["history"] for r in rs]
        rows.append([name, {**DEFAULTS, **grid[name]}, runs, tr.mean(), tr.std(ddof=1),
                     te.mean(), te.std(ddof=1), rs[0]["evals"]])
        print(f"{exp:12s} {name:18s} train {tr.mean():.4f}±{tr.std(ddof=1):.4f}  "
              f"test {te.mean():.4f}±{te.std(ddof=1):.4f}")
    with open(f"results/{exp}_summary.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["name", "config", "runs", "train_mean", "train_sd", "test_mean", "test_sd", "evals"])
        w.writerows(rows)
    with open(f"results/{exp}_hist.json", "w") as f:
        json.dump(hist, f)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("experiment", help="list | all | " + " | ".join(EXPERIMENTS))
    p.add_argument("--runs", type=int, default=10)
    p.add_argument("--workers", type=int, default=os.cpu_count() or 1)
    p.add_argument("--data", default=DATA)
    a = p.parse_args()
    if a.experiment == "list":
        for e, g in EXPERIMENTS.items():
            print(f"{e}: {len(g)} configs -> {list(g)}")
    else:
        for e in (EXPERIMENTS if a.experiment == "all" else [a.experiment]):
            run_experiment(e, a.runs, a.workers, a.data)


In [ ]:
%%writefile plot.py
"""Make figures from sweep.py output.   python plot.py budget   (or: python plot.py all)

Writes results/<exp>_bars.png (mean test/train accuracy, error bars = 1 sd over runs)
and results/<exp>_convergence.png (mean best training fitness vs evaluations).
"""
import csv
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def plot(exp):
    rows = list(csv.DictReader(open(f"results/{exp}_summary.csv")))
    hist = json.load(open(f"results/{exp}_hist.json"))
    names = [r["name"] for r in rows]
    x = np.arange(len(names))

    fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(names) + 2), 4))
    for off, key, lab in ((-0.2, "train", "train"), (0.2, "test", "test")):
        ax.bar(x + off, [float(r[f"{key}_mean"]) for r in rows], 0.4,
               yerr=[float(r[f"{key}_sd"]) for r in rows], capsize=2, label=lab)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_ylabel("accuracy"); ax.set_title(f"{exp}: mean ± sd over runs")
    lo = min(float(r["test_mean"]) for r in rows)
    ax.set_ylim(max(0.0, lo - 0.1), 1.005); ax.legend()
    fig.tight_layout(); fig.savefig(f"results/{exp}_bars.png", dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    for r in rows:
        h = np.array(hist[r["name"]])                         # runs x iterations
        swarm = json.loads(r["config"].replace("'", '"').replace("True", "true")
                           .replace("False", "false"))["swarm"]
        ev = (np.arange(h.shape[1]) + 1) * swarm
        m, s = h.mean(0), h.std(0, ddof=1)
        ax.plot(ev, m, label=r["name"]); ax.fill_between(ev, m - s, m + s, alpha=0.15)
    ax.set_xlabel("fitness evaluations"); ax.set_ylabel("best training accuracy")
    ax.set_title(f"{exp}: convergence"); ax.legend(fontsize=7, ncol=2)
    fig.tight_layout(); fig.savefig(f"results/{exp}_convergence.png", dpi=150); plt.close(fig)
    print("wrote", f"results/{exp}_bars.png", f"results/{exp}_convergence.png")


if __name__ == "__main__":
    import glob
    exps = ([p.split("/")[-1][:-12] for p in glob.glob("results/*_summary.csv")]
            if sys.argv[1] == "all" else [sys.argv[1]])
    for e in exps:
        plot(e)


In [ ]:
%%writefile compare.py
"""Welch's t-test between two configurations of one experiment (uses sweep.py output).

    python compare.py budget swarm10x50it swarm50x10it           # compares TEST accuracy
    python compare.py coef b1.5_g1.5 b0.5_g2.5 train

p < 0.05 suggests the difference in mean accuracy is unlikely to be chance given the
number of runs. With only 10 runs per setting, report it as supporting evidence, not proof.
"""
import csv
import sys

from scipy.stats import ttest_ind_from_stats


def main():
    exp, a, b = sys.argv[1:4]
    key = sys.argv[4] if len(sys.argv) > 4 else "test"
    rows = {r["name"]: r for r in csv.DictReader(open(f"results/{exp}_summary.csv"))}
    for n in (a, b):
        if n not in rows:
            sys.exit(f"'{n}' not found. Available: {list(rows)}")
    s = [(float(rows[n][f"{key}_mean"]), float(rows[n][f"{key}_sd"]), int(rows[n]["runs"])) for n in (a, b)]
    t, p = ttest_ind_from_stats(s[0][0], s[0][1], s[0][2], s[1][0], s[1][1], s[1][2], equal_var=False)
    print(f"{key} accuracy  {a}: {s[0][0]:.4f}±{s[0][1]:.4f}   {b}: {s[1][0]:.4f}±{s[1][1]:.4f}")
    print(f"Welch t = {t:.3f}, p = {p:.4f}  ->  " + ("significant at 0.05" if p < 0.05 else "not significant at 0.05"))


if __name__ == "__main__":
    main()


## Cell 8: check all files exist
You should see: ann.py, compare.py, data.py, data_banknote_authentication.txt, plot.py, pso.py, run.py, sweep.py

In [ ]:
!ls


## Cell 9: test the basic run (expect accuracy near 1.0)

In [ ]:
!python run.py --runs 3


## Cell 10: test the learned-activations extension

In [ ]:
!python run.py --layers 4 4 1 --learn_acts --runs 3


## Cell 11: run one experiment (fixed 500-evaluation budget) and show plots
Other names to try instead of `budget`: `arch`, `coef`, `informants`, `boundary`, `learned_acts`, `metric`.

In [ ]:
!python sweep.py budget
!python plot.py budget
from IPython.display import Image, display
display(Image('results/budget_bars.png'))
display(Image('results/budget_convergence.png'))


## Optional: is the difference between two settings real? (Welch t-test)
Run AFTER Cell 11. Replace the names with two settings from your experiment.

In [ ]:
!python compare.py budget swarm10x50it swarm50x10it


## Cell 12: download your results (do this before closing Colab)

In [ ]:
!zip -r results.zip results
from google.colab import files
files.download('results.zip')
